# 🐍 Chapter 12 — Python UDFs (User Defined Functions) and what they cost

A **UDF** is a function *you* write, handed to Spark so it can be used on a DataFrame column
just like a built-in one. Writing one is easy — and that is the trap. A Python UDF is the one
thing in PySpark that pulls your own Python process into the middle of the data path, so this
chapter is as much a **warning label** as a how-to.

Source: the *UDF* video of the PySpark playlist —
[youtu.be/bbNUiWfAP90](https://www.youtube.com/watch?v=bbNUiWfAP90&list=PL2IsFZBGM_IHCl9zhRVC1EXTomkEp_1zm&index=18)

> ⚠️ **Where to run this chapter.** A UDF needs a working Python on whichever machine executes
> it — so *where* you run matters more here than in any previous chapter. Three environments, not
> interchangeable (this notebook is set up for the third one):
>
> | Environment | Python | UDFs? |
> |-------------|--------|-------|
> | Local `.venv` on Windows | 3.12 + PySpark 3.3.0 | ❌ nothing runs — cloudpickle cannot serialise a function on 3.12, so even `createDataFrame` on a plain list dies with `PicklingError: Could not serialize object` |
> | Docker Jupyter container, `.master("local[*]")` | 3.7.17 | ✅ everything in this chapter, pandas UDFs included |
> | Docker standalone cluster, `.master("spark://bd-spark-master:7077")` | workers on 3.7.10 | ⚠️ plain UDFs only, and only after the worker fix below — pandas UDFs cannot run there |
>
> (**Pickle** = Python's built-in way of turning an object into a stream of bytes so it can be
> stored or sent elsewhere; **unpickle** turns those bytes back into an object.)
>
> **The two cluster-mode gotchas**, both worth understanding because they are what section 2 looks
> like in real life:
> 1. **The python path is baked into the shipped UDF.** The driver's `PYSPARK_PYTHON`
>    (`/usr/local/bin/python` in the Jupyter image) travels with the function, and the executor
>    tries to launch *that exact path*. The Alpine workers only have `/usr/bin/python3`, so every
>    task dies with `Cannot run program "/usr/local/bin/python": error=2, No such file or
>    directory`. Fix: `docker exec bd-spark-worker-1 ln -sf /usr/bin/python3 /usr/local/bin/python`
>    (and the same for worker-2) — that is what the commented-out `ln -s` line in
>    `docker-images/docker-compose.yml` was for.
> 2. **pandas UDFs need `pandas` + `pyarrow` on *every* executor.** The Jupyter image can install
>    them (`pip install pandas==1.3.5 pyarrow==12.0.1`), but the workers are Alpine 3.10 + musl
>    Python 3.7, which has no binary wheels for either — so on this cluster, keep pandas UDFs to
>    `local[*]`.
>
> The `explain()` cells are safe anywhere: they only print a plan, they never execute one.

**How this chapter is laid out:** section 1 is hands-on — you run a cell, then read a short note
directly under it explaining that one line. Sections 2 onwards are the deeper story, once the code
already makes sense.

| # | Section | Question it answers |
|---|---------|---------------------|
| 1 | Writing your first UDF | Notes sit *between* the code cells: what each line did, why `udf()` is needed, what "once per row" means, the return-type trap, and `BatchEvalPython` |
| 2 | The journey of one row | What happens on the worker node when my UDF runs? (+ what *serialise* means) |
| 3 | The four costs | It works — so why is that not good enough? |
| 4 | Memory & OOM | Why does a UDF blow up a node with `OutOfMemoryError`? |
| 5 | The remaining traps | Nulls, pickled closures, determinism, executor Python |
| 6 | Fix 1 — built-ins | Higher-order functions instead of a UDF |
| 7 | Fix 2 — Scala/Java UDF | Reuse a JVM UDF from Python — no python worker at all |
| 8 | Fix 3 — pandas UDF | Still Python, but a whole batch at a time |
| 9 | Cheat sheet | What to reach for, in what order |

---


In [ ]:
from pyspark.sql import SparkSession

# this session runs on the CLUSTER: the driver is this notebook (Jupyter container), the
# executors are JVMs inside the two worker containers. For UDFs that means:
#   - the workers need a python at the driver's PYSPARK_PYTHON path (/usr/local/bin/python):
#       docker exec bd-spark-worker-1 ln -sf /usr/bin/python3 /usr/local/bin/python   (same for -2)
#   - every file you read must sit on the shared /data volume, not /home/jupyter
#   - pandas UDFs will NOT run here (no pandas/pyarrow on the workers) - use .master("local[*]")
#     for that one cell. See the warning box above.
spark = (SparkSession.builder
         .appName("UserDefinedFunctions")
         .master("spark://1d82a0b4d4fd:7077")
         .config("spark.executor.cores", "2") # each executor has 2 cores
         .config("spark.cores.max", "6") # max 6 cores for all executors
         .config("spark.executor.memory", "512m")
         .getOrCreate())
spark
"""3 executors, each with 2 cores, so 6 cores max for all executors. 512m memory per executor."""

In [12]:
# Read employee data
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date date"
emp = spark.read.format("csv").option("header", True).schema(emp_schema).load("/data/csv/emp.csv")
emp.rdd.getNumPartitions()

1

## 📝 1. Writing your first UDF — one cell at a time

The next few cells are the whole job: **a plain Python function, one line to hand it to Spark, and
one line to use it.** Run each cell, then read the short note under it — each note explains only the
line you just ran. The deeper "what happens on the cluster" story is section 2, after this.

---


In [13]:
# Define a UDF to calculate bonus
def bonus(salary):
    return float(salary) * 0.1

#### ☝️ That cell was ordinary Python — Spark has not been told anything yet

`bonus` takes **one** salary and returns **one** number. You can call it right now:
`bonus(50000)` gives `5000.0`. It knows nothing about DataFrames, columns or rows — and Spark
knows nothing about it. It is just a function sitting in your notebook.

---


In [ ]:
from pyspark.sql.functions import udf
bonus_udf = udf(bonus)
# this is available for the dataframe api. if we need to use it in SQL, Spark SQL expression, we need to register it.

#### ☝️ That line is the *registration* — it hands your function to Spark

Why it is needed: your function wants a salary, but all you can hold in notebook code is the
**word** `"salary"`. Call it directly and both attempts fail:

```python
bonus("salary")        # ValueError: could not convert string to float: 'salary'
bonus(col("salary"))   # TypeError: float() argument must be a string or a number, not 'Column'
```

`col("salary")` is not a number — it is a **description**: "the salary column of this DataFrame".

So `udf()` gives you back a *new* thing, `bonus_udf`, which does two jobs your `def` cannot:

- **it takes a Column and returns a Column**, so Spark can drop it into its query plan (the recipe
  Spark builds before running anything), and
- **it packages your function for travel** — pickled, so it can be shipped to the machines that
  actually hold the rows.

Read it as a sentence spoken to Spark: *"here is my function `bonus` — YOU call it."*

---


In [15]:
emp.withColumn("bonus", bonus_udf("salary")).show()
# if you check on the terminal you will see a python process is started spinning up by spark for processing 
# this data using the UDF

+-----------+-------------+-------------+---+------+------+----------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+------+----------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|5000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|4500.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|5500.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|4800.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|6000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|5200.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|7000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|5100.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|5800.0|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-

#### ☝️ What just happened — your function ran 20 times

```text
 emp table              your function           new column
 ---------              -------------           ----------
 salary                                         bonus
  50000      ----->     bonus(50000)   ----->    5000.0     call 1
  45000      ----->     bonus(45000)   ----->    4500.0     call 2
  55000      ----->     bonus(55000)   ----->    5500.0     call 3
  48000      ----->     bonus(48000)   ----->    4800.0     call 4
                             ...
                                                 (20 rows = 20 calls)
```

**Spark wrote the loop, not you** — out on the machines holding the rows:

```text
 for each row:
     value  = row.salary       # "50000" - a real value at last
     result = bonus(value)     # YOUR function, one call per row
     put result into the new "bonus" column for that row
```

20 rows → 20 calls. 10 million rows → 10 million calls. That invisible loop is where the whole cost
of a UDF hides — section 3.

> You already relied on this before today: `regexp_replace(col("name"), "J", "Z")` in chapter 4 also
> ran once per row. A UDF is the same deal, except the function being run is *yours*.

---


In [ ]:
from pyspark.sql.functions import udf
spark.udf.register("bonus_sql_udf", bonus , "double")

<function __main__.bonus(salary)>

#### ☝️ `register` puts the same function somewhere else — the SQL registry

`bonus_udf = udf(bonus)` gave you something usable in the **DataFrame API** only.
`spark.udf.register("bonus_sql_udf", bonus, "double")` adds it to Spark's **function registry**, the
list of function names Spark SQL can resolve. That is what makes the *next* cell work — a name
inside a SQL string.

Reading that line argument by argument — **none of the three is a column**:

```text
 spark.udf.register("bonus_sql_udf", bonus, "double")
                    ^^^^^^^^^^^^^^^  ^^^^^  ^^^^^^^^
                    1                2      3

 1  the NAME that SQL will call it by (quoted - it is just text)
 2  the FUNCTION you wrote with def  (no quotes - an object)
 3  what it RETURNS
```

The cell's own output says it out loud: `<function __main__.bonus(salary)>` — a function, not a
column. Columns only appear when you *use* the UDF: in `expr("bonus_sql_udf(salary)")` the column
going **in** is `salary`, and in `withColumn("bonus", ...)` the column coming **out** is `"bonus"`.
The word "bonus" doing three jobs in this notebook — the function, the UDF name, the new column — is
the whole reason this looks confusing.

Two differences worth spotting between your two lines:

- the registered one has a **name as text** (`"bonus_sql_udf"`), because SQL can only refer to things
  by name;
- the registered one **declares a return type** (`"double"`); your `udf(bonus)` did not. That matters
  more than it looks — see the return-type cell below.

---


In [18]:
from pyspark.sql.functions import expr
emp.withColumn("bonus", expr("bonus_sql_udf(salary)")).show()

# whats happening here is that the UDF is being called on each row of the dataframe, and the result is being added as a new column called "bonus". 
# bonus_sql_udf is the name of the table which contain salary coulumn

+-----------+-------------+-------------+---+------+------+----------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date| bonus|
+-----------+-------------+-------------+---+------+------+----------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|5000.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|4500.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|5500.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|4800.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|6000.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|5200.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|7000.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|5100.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|5800.0|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-

#### ☝️ One correction to the comment in that cell

Your comment says *"bonus_sql_udf is the name of the table which contain salary column"*. It is
**not a table** — it is the **name you gave your function** in
`spark.udf.register("bonus_sql_udf", …)`.

Inside `expr("bonus_sql_udf(salary)")`:

- `bonus_sql_udf` → your function, looked up in the SQL registry
- `salary` → the column being passed into it

No table is named in that string at all, because `emp` is already decided by `emp.withColumn(...)`.
You can see your function listed with `spark.catalog.listFunctions()` — the cell below prints it.

---


In [ ]:
# same without using UDF
emp.withColumn("bhonus",expr("salary * .01")).show()

+-----------+-------------+-------------+---+------+------+----------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date|bhonus|
+-----------+-------------+-------------+---+------+------+----------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01| 500.0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15| 450.0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01| 550.0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30| 480.0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01| 600.0|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01| 520.0|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15| 700.0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01| 510.0|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01| 580.0|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-

#### ☝️ And that cell is the punchline of the whole chapter

No `def`. No registration. No Python process on the workers. The same arithmetic, done by Spark's
own engine — and the column comes out as a proper `double`.

`bonus()` never needed to be a UDF, and that is the **normal** case, not the exception. Section 6
lists what to check before writing one. (Your two cells use different rates — `* 0.1` in the UDF,
`* .01` here — so the numbers differ 10×; the *mechanism* is what is being compared.)

---


In [ ]:
# the DECORATOR form - the third way to declare the same UDF
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

@udf(returnType=DoubleType())          # the return type your bonus_udf left out
def bonus_decorated(salary):
    return float(salary) * 0.1

emp.withColumn("bonus", bonus_decorated("salary")).select("name", "salary", "bonus").show(5)


#### ☝️ Same UDF, third spelling — the three ways side by side

| How you write it | Where you can call it | Return type |
|---|---|---|
| `bonus_udf = udf(bonus)` | DataFrame API only | not declared → **string** |
| `@udf(returnType=DoubleType())` on the `def` | DataFrame API only | declared → double |
| `spark.udf.register("bonus_sql_udf", bonus, "double")` | SQL **and** DataFrame API | declared → double |

The decorator form is just `udf()` written on top of the function instead of on a separate line.
After it, the name `bonus_decorated` **is** the UDF — the plain Python function is no longer
directly callable under that name.

---


In [ ]:
# the return-type trap, on your own two registrations
from pyspark.sql.functions import udf, expr

print(">>> udf(bonus) - no type declared, so Spark assumed StringType")
emp.withColumn("bonus", bonus_udf("salary")).select("bonus").printSchema()

print(">>> register(..., 'double') - the SAME function, a real number")
emp.withColumn("bonus", expr("bonus_sql_udf(salary)")).select("bonus").printSchema()

# and the worst case: declare a type the function does not actually return
wrong_udf = udf(bonus, "int")          # bonus returns 5000.0, a float
print(">>> udf(bonus, 'int') - silent nulls, no error raised")
emp.withColumn("bonus", wrong_udf("salary")).select("name", "salary", "bonus").show(3)

# your registered name really is in the SQL registry - not a table:
print([f.name for f in spark.catalog.listFunctions() if "bonus" in f.name])


#### ☝️ The return-type trap, in your own two cells

Spark works out the schema of the result **before** a single row is read, and it cannot look inside
your Python function to guess. So the type is on you:

- **You left it out** in `udf(bonus)` → Spark assumed `StringType`, and your `bonus` column is the
  *text* `"5000.0"`, not a number.
- **You declared it** in `register(..., "double")` → a real `double`.
- **Declare the wrong one** — `udf(bonus, "int")` on a function returning `5000.0` — and every row
  comes back **`null`**, with no exception and no warning at all.

Both `show()` outputs look identical, because `show()` prints everything as text. The difference
only bites later: a string column sorts alphabetically, aggregates differently, and writes a
different type to parquet.

---


In [ ]:
# your UDF vs your built-in, seen in the query plan
from pyspark.sql.functions import expr

print(">>> your UDF - look for BatchEvalPython")
emp.withColumn("bonus", bonus_udf("salary")).explain()

print(">>> your built-in expr - no BatchEvalPython anywhere")
emp.withColumn("bhonus", expr("salary * .01")).explain()

# BatchEvalPython [bonus(salary)] is the step where rows leave the JVM, cross into the python
# worker and come back. Spotting it in a plan is the standard way to find a slow UDF.


#### ☝️ `BatchEvalPython` is the UDF's fingerprint in the plan

`explain()` prints the physical plan — the steps Spark will actually run. In your UDF version there
is a `BatchEvalPython [bonus(salary)]` line; in the `expr("salary * .01")` version there is nothing
of the sort.

That one line **is** the detour: rows leaving the JVM, crossing into a Python process, and coming
back. Section 2 is what happens inside it. Spotting `BatchEvalPython` in a plan is the standard way
to find a slow UDF someone left in a pipeline.

---


## 📝 2. The journey of one row — what really happens when a UDF runs

**Look at the picture first. Every number in it is explained underneath, in the same order.**

```text
 ┌────────────────────────────────────────────┐
 │ DRIVER  (your notebook)                    │
 │ 1. you call the UDF - nothing runs yet     │
 │ 2. on the action: pickle your function,    │
 │    ship it with every task                 │
 └───────────────────┬────────────────────────┘
                     │  the CODE travels to the data
 ┌───────────────────▼────────────────────────┐
 │ EXECUTOR JVM   (1 task = 1 partition)      │
 │ 3. start a python worker process           │
 │ 4. serialise a batch of rows out           │
 └───────────────────┬────────────────────────┘
                     │  local socket
 ┌───────────────────▼────────────────────────┐
 │ PYTHON WORKER  (separate OS process)       │
 │ 5. run YOUR function - ONE ROW AT A TIME   │
 └───────────────────┬────────────────────────┘
                     │  local socket
 ┌───────────────────▼────────────────────────┐
 │ EXECUTOR JVM                               │
 │ 6. deserialise results, finish the plan    │
 │ 7. report the task result to the DRIVER    │
 └────────────────────────────────────────────┘
```

#### The two words you need for the list below

**Serialise = flatten something that lives in memory into a plain row of bytes, so it can travel.
Deserialise = rebuild it from those bytes on the other side.**

Why it is needed at all: **two processes cannot share memory.** The executor JVM holds your row as
a Java object at some memory address; the python worker is a *different* process and cannot reach
into that memory. The only thing that can cross between two processes is a stream of bytes down a
socket.

```text
 in the JVM                on the wire             in Python
 ----------                -----------             ---------
 Row object      --pack--> 01001010 0110 --unpack-> ("001","John Doe")
 (Java memory)             just bytes,              (Python objects)
                           no structure
    serialise                                        deserialise
```

Think of flat-pack furniture: you cannot post an assembled wardrobe, so you take it apart into flat
panels, post it, and rebuild it at the far end. The wardrobe was fine to begin with — the packing
and unpacking is pure overhead you pay only because it has to cross a boundary.

In the steps below it happens **twice**: step 4 packs rows on the way out, step 6 unpacks results on
the way back. That is the whole of Cost 1 in section 3.

Now the same seven numbers, in words:

1. **You call the UDF inside a transformation** — `emp.withColumn("bonus", bonus_udf("salary"))`. Nothing runs;
   Spark only records it in the plan (**lazy evaluation**: transformations build the plan, an action
   such as `show()` / `count()` / `write` triggers it).
2. **On the action, the driver ships your logic to the workers.** Spark *pickles* your function
   together with its **closure** (any variable from outside the function that the function uses) and
   sends those bytes along with every task. This is what "Spark copies the logic to the worker nodes"
   means — the **code travels to the data**, never the other way round.
3. **The executor starts a python worker process.** An executor is a **JVM** (Java Virtual Machine —
   the process that runs Java/Scala bytecode). Your Python cannot run *inside* a JVM, so the executor
   launches a **separate operating-system process** on the same node (`python -m pyspark.daemon`),
   one per concurrently running task slot. It is reused by later tasks, but the first one pays the
   startup cost.
4. **Rows are serialised out of the JVM.** **Serialise** = turn objects in memory into a flat stream
   of bytes so another process can read them. JVM row objects → bytes → local socket → the python
   worker **deserialises** them back into Python objects.
5. **Your function runs one row at a time.** One Python call per row — the loop drawn in section 1's notes. No
   batching, no vectorising, and none of the compiled JVM code Spark generates for built-ins.
6. **Results are serialised back.** Python objects → bytes → socket → the JVM deserialises them into
   rows, and the rest of the plan (filters, joins, aggregation, writing) continues inside the JVM.
7. **The JVM reports the finished task to the driver**, exactly as it would for any other task. The
   driver never talks to the python worker.

### Where those processes actually sit

The figure above is the *sequence*. This one is the *map* — which box lives on which machine:

```text
 ┌───────────────────────────────────┐
 │ DRIVER PROGRAM                    │
 │  ┌────────────┐   ┌────────────┐  │
 │  │ JVM        │←──│ Python     │  │
 │  │ plan +     │   │ your code  │  │
 │  │ scheduling │   │ (notebook) │  │
 │  └────────────┘   └────────────┘  │
 └─────────────────┬─────────────────┘
                   │ tasks + the pickled UDF
 ┌─────────────────▼─────────────────┐
 │ CLUSTER                           │
 │  ┌─────────────────────────────┐  │
 │  │ NODE 1                      │  │
 │  │  ┌──────────┐ ┌──────────┐  │  │
 │  │  │ JVM      │ │ python   │  │  │
 │  │  │ executor │ │ worker   │  │  │
 │  │  └──────────┘ └──────────┘  │  │
 │  └─────────────────────────────┘  │
 │  ┌─────────────────────────────┐  │
 │  │ NODE 2                      │  │
 │  │  ┌──────────┐ ┌──────────┐  │  │
 │  │  │ JVM      │ │ python   │  │  │
 │  │  │ executor │ │ worker   │  │  │
 │  │  └──────────┘ └──────────┘  │  │
 │  └─────────────────────────────┘  │
 └───────────────────────────────────┘
```

Two things to take from this map:

- **Your Python code exists in two different places** — the one you type in (the driver), and one
  *per executor* out on the cluster. Same function, different processes.
- **The python worker sits *beside* the executor JVM, not inside it.** Same node, same box in the
  picture, separate process. That single detail is the cause of everything in sections 3 and 4.

---


## 📝 3. The four costs of a Python UDF

Every one of these comes straight out of the seven steps above.

#### Cost 1 — serialisation, twice, for every row

Data that was already sitting in the executor's memory has to be converted to bytes, pushed through
a socket, rebuilt as Python objects, then converted back the other way. A built-in function does
none of this: the rows never leave the JVM.

#### Cost 2 — an extra process per executor

The executor must launch and feed a python worker. It costs startup time, and it costs memory that
your `--executor-memory` setting never accounted for (section 4).

#### Cost 3 — one row at a time

Spark's built-ins are compiled into JVM code that runs over whole columns. A Python UDF is a Python
function call per row — interpreted, one at a time. On millions of rows the difference is not a few
percent, it is a different order of magnitude.

#### Cost 4 — the optimiser goes blind

Catalyst treats the UDF as an opaque box, which costs you more than speed:

- **No pushdown.** `emp.filter(my_udf(col("x")) == "y")` cannot be pushed into the file scan or the
  database, so Spark reads everything and filters afterwards. `col("x") == "y"` would have been
  pushed down.
- **No reordering, no folding.** Spark cannot decide the UDF is cheap enough to run early or
  expensive enough to run last — it has no idea what is inside.
- **It can run more often than you think.** Spark may evaluate a UDF before the condition you
  assumed protects it, so `when(col("x").isNotNull(), my_udf("x"))` does **not** guarantee your
  function never sees a `None`. Guard inside the function.

---


## 📝 4. Why a UDF can take the whole node down with OutOfMemory

**Spark sizes and tracks the memory of the executor *JVM* — the python worker is a different
process, so the memory your Python code uses is outside everything Spark measures.**

```text
 ONE WORKER NODE  (say 8 GB of RAM)
 ┌────────────────────────────────────────────────┐
 │ executor JVM                                   │
 │  ┌──────────────────────────────────────────┐  │
 │  │ heap = --executor-memory 4g              │  │
 │  │ Spark tracks and manages every byte here │  │
 │  └──────────────────────────────────────────┘  │
 │                                                │
 │ python workers   (OUTSIDE the heap)            │
 │  ┌──────────────────────────────────────────┐  │
 │  │ your dict / model / pandas frame / cache │  │
 │  │ not measured, not capped by default      │  │
 │  └──────────────────────────────────────────┘  │
 └────────────────────────────────────────────────┘
   heap 4g  +  python growing with no cap
   ---> the node runs out of RAM
   ---> the OS kills the process ---> the task fails
```

What the picture is saying, line by line:

- **`--executor-memory 4g` sets the inner box only.** *Heap* = the memory region a JVM manages and
  garbage-collects. Spark knows how much of it is used, spills to disk when it fills, and reports it
  in the UI.
- **The python worker is the second box — not in that budget.** It is an ordinary OS process taking
  RAM from whatever the node has left. On YARN or Kubernetes that space is
  `spark.executor.memoryOverhead`; in standalone mode it is simply the machine's free memory.
- **Spark cannot limit the second box by default.** The JVM cannot garbage-collect another process'
  memory. The one dial that exists is `spark.executor.pyspark.memory` — **unset by default**, i.e.
  uncapped as far as Spark is concerned. (The video's "Spark has no control over the Python process"
  is the right idea; precisely, Spark can start it, stop it and — if you set that config — cap it,
  but it can never manage what is inside it.)
- **What makes it grow:** loading a big lookup dictionary, a model, or a pandas frame inside the
  function; accumulating state in a global; a library that caches. Multiply by the number of task
  slots on the node — every concurrent task has its own python worker.
- **The failure doesn't look like a Spark error.** The OS (or the container runtime) kills the
  process, and you see `Python worker exited unexpectedly (crashed)`, a lost executor, or a
  container `Killed by YARN for exceeding memory limits` — with no useful Python traceback.

---


## 📝 5. The remaining traps

Beyond the return type, four things bite people who write UDFs:

- **`None` will arrive.** SQL `NULL` becomes Python `None`. Your `float(salary)` would raise
  `TypeError` on a null salary — and one raised exception fails the task, then after
  `spark.task.maxFailures` retries, the whole job. `emp.csv` has no nulls; a real file will.
- **Everything the function touches gets pickled.** A large object referenced from the enclosing
  scope is shipped with *every task*; use `spark.sparkContext.broadcast(obj)` to send one read-only
  copy per executor instead. A `SparkSession` or DataFrame referenced inside a UDF cannot be pickled
  at all — a UDF runs on the executor, where there is no session.
- **Spark assumes a UDF is deterministic** (same input → same output) and may reorder or re-run it.
  If it isn't (random, time, a network call), mark it `udf(...).asNondeterministic()`.
- **The executors need the same Python you have, at the same path, with the same libraries.** The
  driver's `PYSPARK_PYTHON` path is *baked into the shipped function*, and the executor launches
  exactly that path. Since your session runs on `spark://`, this is live for you — see the warning
  box at the top of the notebook.

---


## 📝 6. Fix 1 (the best one) — use built-ins, including *higher-order* functions

**The first answer to "should I write a UDF?" is almost always "no — a built-in already does this".**
Built-ins are executed by Spark's own engine inside the JVM: no python worker, no serialisation, and
Catalyst can still optimise around them.

**You already proved this yourself.** These two cells produce the same numbers:

```python
emp.withColumn("bonus",  bonus_udf("salary"))        # your UDF - a trip to a python worker
emp.withColumn("bhonus", expr("salary * .01"))       # your built-in - never leaves the JVM
```

The second one needs no `def`, no registration, no python process on the workers, and it keeps the
column as a proper `double`. `bonus()` was never a job for a UDF — that is the normal case, not the
exception. (Note your two cells use different rates — `* 0.1` in the UDF, `* .01` in the built-in —
so the numbers differ by 10×; the *mechanism* is what is being compared.)

#### What "higher-order function" means

**A higher-order function is a function that takes another function as one of its arguments.** In
Spark that's the array functions — `transform`, `filter`, `exists`, `aggregate` — where you pass a
small lambda: `transform(nums, x -> x * 2)`.

The important part: **that lambda is a Spark SQL expression, not Python code.** Even when you write
it as a Python `lambda` in PySpark, you are building a Column expression that Spark compiles into
the plan — it is never pickled and never shipped to a python worker. So "loop over the elements of
an array and change each one", the classic reason people reach for a UDF, is a built-in operation
that costs nothing extra.

---


In [ ]:
# 6) higher-order functions on an array column - the lambda runs in the JVM
'''SELECT nums,
          transform(nums, x -> x * 2)              AS doubled,
          filter(nums, x -> x > 2)                 AS big,
          aggregate(nums, 0, (acc, x) -> acc + x)  AS total
   FROM nums_tbl'''
from pyspark.sql import functions as F

nums = spark.createDataFrame([(1, [1, 2, 3, 4]), (2, [5, 6])], "id int, nums array<int>")

# written as SQL strings ...
nums.select(
    "nums",
    F.expr("transform(nums, x -> x * 2)").alias("doubled"),
    F.expr("filter(nums, x -> x > 2)").alias("big"),
    F.expr("aggregate(nums, 0, (acc, x) -> acc + x)").alias("total"),
).show(truncate=False)

# ... or with PySpark's wrappers, where the Python lambda BUILDS a Column expression
# (it is evaluated once, at plan time - it is not shipped to a python worker)
nums.select(
    "nums",
    F.transform("nums", lambda x: x * 2).alias("doubled"),
    F.filter("nums", lambda x: x > 2).alias("big"),
    F.aggregate("nums", F.lit(0), lambda acc, x: acc + x).alias("total"),
).show(truncate=False)


## 📝 7. Fix 2 — write the UDF in Scala/Java and call it from PySpark

**If the logic really has no built-in, write it in the JVM's own language: a Scala or Java UDF is
registered *inside* the executor JVM, so calling it from PySpark starts no python worker and
serialises nothing.** Your Python code just names it.

Why this works: the barrier is not "Python the language", it is the **process boundary**. A Scala
UDF lives in the same process as the data, so it is an ordinary function call — same cost class as
a built-in (it is still opaque to Catalyst, so Cost 4 from section 3 still applies).

**Step 1 — write your `bonus` in Scala** (taking a `String`, since that is how `salary` is typed in
your schema):

```scala
package com.example.udfs

import org.apache.spark.sql.api.java.UDF1

class Bonus extends UDF1[String, java.lang.Double] {
  override def call(salary: String): java.lang.Double = {
    if (salary == null) null else salary.toDouble * 0.1
  }
}
```

**Step 2 — build a jar** (`sbt package` / `mvn package`). A **jar** is a zip of compiled JVM
classes — the unit of code you hand to Spark.

**Step 3 — give the jar to the session**, so the class exists in the driver *and* every executor:

```bash
spark-submit --jars /path/udfs.jar my_job.py
# or, from a notebook, before the session is created:
#   SparkSession.builder.config("spark.jars", "/path/udfs.jar")
```

On this cluster the jar must sit somewhere all containers can reach — the `/data` volume, the same
place your `emp.csv` had to move to.

**Step 4 — register the class from Python and use it by name** — see the next cell.

---


In [ ]:
# calling a Scala/Java UDF from PySpark  (template - needs the jar from step 2/3)
'''SELECT name, salary, bonus_scala_udf(salary) AS bonus FROM emp'''

# spark.udf.registerJavaFunction(
#     "bonus_scala_udf",            # the SQL name you will call it by
#     "com.example.udfs.Bonus",     # fully-qualified class name inside the jar
#     "double",                     # return type (optional; Spark can infer it from UDF1[..])
# )
#
# emp.withColumn("bonus", expr("bonus_scala_udf(salary)")).show(5)
# spark.sql("SELECT name, salary, bonus_scala_udf(salary) AS bonus FROM emp").show(5)
#
# The plan for this has NO BatchEvalPython step - the work happens inside the executor JVM,
# so no python worker is ever started on the workers.
# Note the asymmetry: from Python you can only call it BY NAME through the SQL registry
# (registerJavaFunction), which is why the expr()/spark.sql() style shows up in every example.


## 📝 8. Fix 3 — pandas UDFs, when it has to be Python

**A pandas UDF (also called a vectorised UDF) is still your Python function on the executor, but
Spark hands it a whole *batch* of rows as a pandas `Series` instead of calling it once per row.**

- **It fixes Cost 1 and Cost 3, not the others.** The transfer uses **Apache Arrow** — a columnar
  in-memory format that both the JVM and Python understand, so a batch is moved with almost no
  conversion work instead of pickling row by row. And your code runs once per batch, letting pandas
  and NumPy do the loop in compiled code.
- **The python worker is still there**, still outside the JVM heap — section 4's memory story applies
  unchanged, and in fact a whole batch now sits in Python memory at once
  (`spark.sql.execution.arrow.maxRecordsPerBatch`, default 10000, is the dial).
- **Catalyst is still blind to it** — Cost 4 stands.
- **Requirements:** `pyarrow` and `pandas` installed on the driver *and* every executor. Type hints
  are how Spark 3.x knows which flavour you mean: `pd.Series -> pd.Series` is a scalar pandas UDF.
- **Rule of the return value:** the Series you return must be the same length as the one you got.

> 🐳 **In this repo:** the Docker Jupyter image ships neither library, so install them there once —
> `docker exec bd-pyspark-jupyter-lab pip install pandas==1.3.5 pyarrow==12.0.1` (those are the last
> versions supporting its Python 3.7). That covers `local[*]`, where the driver *is* the executor.
> It does **not** extend to `spark://` runs: the Alpine/musl workers have no wheels available, so a
> pandas UDF there fails with a `PythonException` from the executor. And the install is lost the
> moment the container is recreated — a two-line Dockerfile makes it stick.

So the ladder is: built-in → higher-order function → Scala/Java UDF → pandas UDF → plain Python UDF.

---


In [ ]:
# the same bonus, as a vectorised pandas UDF
# NOTE: this one needs .master("local[*]") in cell 1 - the Alpine workers have no pandas/pyarrow,
#       so on spark:// it fails with a PythonException. See section 8.
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")                                  # return type as a DDL string
def bonus_vec(salary: pd.Series) -> pd.Series:         # a WHOLE COLUMN of a batch, not one value
    return salary.astype(float) * 0.1                  # vectorised: pandas loops in compiled code

emp.withColumn("bonus", bonus_vec("salary")).select("name", "salary", "bonus").show(5)

# in the plan this shows as ArrowEvalPython (instead of BatchEvalPython) - still a trip out
# to the python worker, but an Arrow-shaped, batch-at-a-time one
emp.withColumn("bonus", bonus_vec("salary")).explain()


## 📝 9. Cheat sheet

**The demerits of a plain Python UDF, in one list:** rows are serialised out and deserialised back;
an extra python process must be started and fed; your code runs one row at a time; Catalyst cannot
optimise around it; and its memory sits outside the JVM heap where Spark can neither see nor cap it.

| Reach for | When | Cost |
|-----------|------|------|
| **Built-in function** (`when`, `regexp_replace`, `to_date`, `split`, …) | Almost always — check the docs first | Runs inside the JVM, fully optimisable |
| **Higher-order function** (`transform`, `filter`, `exists`, `aggregate`) | Per-element work on an `array<...>` column | Same as a built-in — the lambda is a plan expression, not Python |
| **Scala/Java UDF** + `registerJavaFunction` | Logic with no built-in, on a hot path, and you can build a jar | JVM-native call; opaque to Catalyst but nothing else |
| **pandas UDF** (`@pandas_udf`) | It must be Python (a NumPy/pandas/ML library call) | Arrow + batch-at-a-time; python worker still outside the heap |
| **Python UDF** (`@udf`) | Small data, prototyping, or logic that genuinely cannot be vectorised | Everything in the list above |

**Handy checks**

- `df.explain()` → `BatchEvalPython` = a Python UDF is in the plan; `ArrowEvalPython` = a pandas UDF;
  neither = it all stays in the JVM.
- `spark.catalog.listFunctions()` → what is registered in the SQL registry (yours included).
- `spark.executor.pyspark.memory` → the only Spark-side cap on python worker memory; unset = uncapped.

---


In [20]:
# Stop Spark Session
spark.stop()
